# Phase 1.6: Baseline Models

Implement and evaluate Popularity and Cosine Similarity baselines.

In [ ]:
import sys
sys.path.insert(0, '../../')

import pandas as pd
import numpy as np
from config import *
from src.data.cache import DataCache
from src.models.baselines import PopularityBaseline, CosineSimilarityBaseline
from src.evaluation.metrics import recall_at_k, ndcg_at_k, mean_reciprocal_rank, aggregate_metrics

cache = DataCache(str(CACHE_DIR))
businesses = cache.load_dataframe('businesses_major_cities')
train_reviews = cache.load_dataframe('train_reviews')
test_reviews = cache.load_dataframe('test_reviews')
test_pairs_meta = cache.load_metadata('test_pairs')
test_pairs = test_pairs_meta['test_pairs']
test_candidates = test_pairs_meta['candidates']

In [ ]:
# Initialize baselines on history reviews (held-out cities already removed)
history_reviews = pd.concat([train_reviews, test_reviews], ignore_index=True)
popularity = PopularityBaseline(businesses, history_reviews)
cosine_sim = CosineSimilarityBaseline(businesses, history_reviews)

def evaluate(model):
    results = []
    for user_id, held_out_city in test_pairs:
        rankings = model.recommend(user_id, test_candidates[user_id], k=PHASE1_EVAL_K)
        results.append({
            'recall@k': recall_at_k(held_out_city, rankings, PHASE1_EVAL_K),
            'ndcg@k': ndcg_at_k(held_out_city, rankings, PHASE1_EVAL_K),
            'mrr': mean_reciprocal_rank(held_out_city, rankings, PHASE1_EVAL_K)
        })
    return aggregate_metrics(results, k=PHASE1_EVAL_K)

print(f"Evaluating on {len(test_pairs)} test pairs...")

In [ ]:
# Evaluate Popularity baseline
pop_metrics = evaluate(popularity)
print("Popularity baseline:")
print(pop_metrics)

In [ ]:
# Evaluate Cosine Similarity baseline
cosine_metrics = evaluate(cosine_sim)
print("Cosine Similarity baseline:")
print(cosine_metrics)

In [ ]:
results = pd.DataFrame([pop_metrics, cosine_metrics], index=['Popularity', 'Cosine Similarity'])
print(results)
print("✓ Phase 1 baseline evaluation complete")